# T11 It1: a candidate judge trained on satellite imagery only

**Rule (team, 2026-11-14):** learned components are trained on satellite imagery only; thermal photos (NUDT, official) may set cut thresholds
and test. **Training data:** 400 drone-view photos rendered from Cartosat scene B and Pleiades Neo (not the map), run through the unchanged
pipeline (`Sat_Photo_Dump`), every candidate labelled by its known true place. **Judges:** S1 = the T7 architecture (monotone GBM, 19 signals);
S5 = relative inliers + scale augmentation (T10 V5). **Cut:** Learn-then-Test (alpha 0.005, floor 0.9) on the NUDT design sets (101-606), with
P from the satellite-trained judge (thermal data only sets the number). **Check:** official TRY (and the NUDT design sets, which only set the
cut). Official TEST, NUDT 707 / 808 are seen data; NUDT 909 is the fresh final set (It2, after pre-registration).
Reference: the NUDT-trained judge (T10), which may not ship.

In [1]:
import os, sys, time, gc
os.environ.setdefault("OMP_NUM_THREADS", "4")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
import train.checks.candidate_judge as cj
import train.checks.scale_robust as sr
from train.checks.decision_signals import build_signal_table
from train.checks.holdout import NUDT_DESIGN, answer_change, bar_frontier, load_t8_table, versus_frontier
from train.checks.eval_tools import manifest, save_manifest
R = ROOT / "tests/Test_Iteration_11/results"; (R / "signals").mkdir(exist_ok=True)
out = R / "signals" / "sat_train.csv.gz"
if not out.exists():
    build_signal_table(R / "rich_dump" / "sat_train", "sat", 4).to_csv(out, index=False)
SAT = pd.read_csv(out)
items = pd.read_csv(R / "sat_photos" / "sat_train" / "items.csv")
from sklearn.cluster import KMeans
fold = KMeans(5, random_state=0, n_init=10).fit_predict(items[["east", "north"]].to_numpy())     # 5 groups of nearby places
SAT["data"] = ["sat%d" % fold[p] for p in SAT.photo]
SAT_SETS = sorted(SAT.data.unique())
SAT = cj.add_context(cj.add_features(SAT))
D = load_t8_table(ROOT, locked=True)                          # NUDT design + official TRY
print(SAT.photo.nunique(), "satellite-made photos;", SAT.groupby("data").photo.nunique().to_dict())

378 satellite-made photos; {'sat0': 64, 'sat1': 94, 'sat2': 78, 'sat3': 93, 'sat4': 49}


## 1. What the satellite-made candidates look like next to thermal ones (labelled candidates, >= 3 inliers)

In [2]:
lab = lambda t: t[cj.labelled(t)].assign(right=lambda d: d.err_sigma <= 4)
cmp = pd.concat([lab(SAT).assign(src="satellite-made"), lab(D[D.data.isin(NUDT_DESIGN)]).assign(src="NUDT design"), lab(D[D.data == "official_try"]).assign(src="official TRY")])
cmp.groupby(["src", "right"])[["inliers", "z", "hull", "cover9", "zgap", "ligap", "angle", "shift_px"]].median().round(3)

inliers      z   hull  cover9   zgap  ligap  angle  shift_px
src            right                                                              
NUDT design    False      4.0  1.036  0.004   0.222 -1.254 -1.723  2.920     6.779
               True      41.0  2.646  0.136   0.714  1.237  3.258  4.104     2.985
official TRY   False      4.0  1.105  0.003   0.222 -1.270 -1.253  2.760     6.757
               True      22.0  2.784  0.138   0.556  1.233  2.996  3.284     3.951
satellite-made False      4.0  1.148  0.004   0.222 -1.268 -1.958  3.188     7.381
               True      52.0  2.220  0.248   0.778  0.522  3.810  3.893     3.305

## 2. Judges trained on satellite-made candidates only; cut on the NUDT design sets

In [3]:
F = cj.FEATURES_CTX; FR = [("rli" if f == "li" else "rln22" if f == "ln22" else f) for f in F]; AUG = [0.5, 0.75, 1.0, 1.5, 2.0]
def judges(m):
    sat = SAT if m is None else cj.add_context(SAT, m=m)
    ev = D if m is None else cj.add_context(D, m=m)
    J = {}
    _, s1 = cj.judge_probability(sat, SAT_SETS, F, "gbm"); J["S1 sat-trained"] = (ev, sr.predict(s1, ev, F))
    lv = sr.far_levels(sat, m); prior = dict(far_inl=float(lv.far_inl.mean()), far_n22=float(lv.far_n22.mean()))
    s5, _, st = sr.fit_judge(sat, SAT_SETS, FR, {"rli", "z", "hull", "cover9"}, AUG, lambda x: sr.add_relative(x, prior, 30, m)); del st; gc.collect()
    evr = sr.add_relative(ev, prior, 30, m); J["S5 sat-trained, relative + augmentation"] = (evr, sr.predict(s5, evr, FR))
    pn, _ = cj.judge_probability(ev, NUDT_DESIGN, F, "gbm"); J["reference: NUDT-trained (not shippable)"] = (ev, pn)
    return J, prior
rows, PRIORS, RESULTS = [], {}, {}
for var, m in [("pc", None), ("jetson", 5)]:
    J, PRIORS[var] = judges(m)
    kw = dict(min_choice=3, wide_tau=0.95, override=None, **(dict(jetson=True, m=5, early_tau=0.98) if m else {}))
    base = cj.baseline(D, 8, 11, min_choice=3) if var == "pc" else cj.baseline(D, 8, 8, jetson=True, m=5)
    RESULTS[(var, "base")] = base
    for name, (ev, p) in J.items():
        dmask = ev.data.isin(NUDT_DESIGN).to_numpy()
        tau = cj.ltt_tau(cj.scan_tau(ev[dmask], p[dmask], cj.TAUS, **{k: v for k, v in kw.items() if k != "early_tau"}), NUDT_DESIGN, 0.005)
        tau = cj.NO_ANSWER if tau is None else max(tau, 0.9)
        r = cj.decide(ev, p, tau, **kw); RESULTS[(var, name)] = r
        s = cj.summary(r); a = answer_change(base, r).set_index("data")
        for d in ["official_try"] + NUDT_DESIGN:
            rows.append(dict(var=var, judge=name, cut=tau, data=d, score=s.loc[d, "score"], wrong=int(s.loc[d, "wrong"]), answered=int(s.loc[d, "n_answered"]),
                             queries=s.loc[d, "queries"], gain=cj.paired_gain(base, r, [d])[d], added_right=a.loc[d, "added_right"], added_wrong=a.loc[d, "added_wrong"]))
    sb = cj.summary(base)
    for d in ["official_try"] + NUDT_DESIGN:
        rows.append(dict(var=var, judge="base", cut=None, data=d, score=sb.loc[d, "score"], wrong=int(sb.loc[d, "wrong"]), answered=int(sb.loc[d, "n_answered"]), queries=sb.loc[d, "queries"]))
SJ = pd.DataFrame(rows); SJ.to_csv(R / "it1_sat_judge.csv", index=False)
SJ[SJ.data == "official_try"].drop(columns="data")

,var,judge,cut,score,wrong,answered,queries,gain,added_right,added_wrong
0,pc,S1 sat-trained,inf,0.000,0,0,0.000,"-51.37 [-54.16, -48.32]",0.0,0.0
6,pc,"S5 sat-trained, relative + augmentation",inf,0.000,0,0,0.000,"-51.37 [-54.16, -48.32]",0.0,0.0
12,pc,reference: NUDT-trained (not shippable),0.9,54.032,0,171,8.270,"+2.66 [+1.17, +4.48]",9.0,0.0
18,pc,base,NaN,51.373,0,162,8.951,NaN,NaN,NaN
24,jetson,S1 sat-trained,inf,0.000,0,0,0.000,"-51.37 [-54.16, -48.32]",0.0,0.0
30,jetson,"S5 sat-trained, relative + augmentation",inf,0.000,0,0,0.000,"-51.37 [-54.16, -48.32]",0.0,0.0
36,jetson,reference: NUDT-trained (not shippable),0.9,51.503,0,167,2.341,"+0.13 [-1.59, +2.01]",5.0,0.0
42,jetson,base,NaN,51.373,0,162,6.530,NaN,NaN,NaN


In [4]:
SJ.pivot_table(index=["var", "judge"], columns="data", values=["score", "wrong"]).round(2)

score                                                wrong                                             
data                                           nudt101 nudt202 nudt404 nudt505 nudt606 official_try nudt101 nudt202 nudt404 nudt505 nudt606 official_try
var    judge                                                                                                                                            
jetson S1 sat-trained                             0.00    0.00    0.00    0.00    0.00         0.00       0       0       0       0       0            0
       S5 sat-trained, relative + augmentation    0.00    0.00    0.00    0.00    0.00         0.00       0       0       0       0       0            0
       base                                      54.24   55.26   55.44   54.68   54.42        51.37       0       0       0       1       2            0
       reference: NUDT-trained (not shippable)   54.64   54.48   57.66   55.12   54.50        51.50       0       0       0       0       1            0
pc     S1 sat-trained                             0.00    0.00    0.00    0.00    0.00         0.00       0       0       0       0       0            0
       S5 sat-trained, relative + augmentation    0.00    0.00    0.00    0.00    0.00         0.00       0       0       0       0       0            0
       base                                      54.84   57.06   54.24   54.24   55.08        51.37       0       0       2       1       2            0
       reference: NUDT-trained (not shippable)   55.52   55.86   56.52   56.56   56.40        54.03       0       1       0       0       1            0

## 3. Is P(right) of the satellite-trained judge honest on thermal photos? (wrong rate by P band)

In [5]:
bins = [0, 0.5, 0.8, 0.9, 0.95, 0.98, 1.0001]
J, _ = judges(None)
rows = []
for name, (ev, p) in J.items():
    L = ev.assign(p=p)[cj.labelled(ev)]
    L = L.assign(band=pd.cut(L.p, bins, right=False), wrong=L.err_sigma > 8, group=np.where(L.data.isin(NUDT_DESIGN), "NUDT design", L.data))
    g = L.groupby(["group", "band"], observed=True).agg(n=("wrong", "size"), wrong_rate=("wrong", "mean")).reset_index().assign(judge=name)
    rows.append(g)
CAL = pd.concat(rows); CAL.to_csv(R / "it1_calibration.csv", index=False)
CAL.pivot_table(index=["judge", "band"], columns="group", values="wrong_rate").round(3)

group                                                 NUDT design  official_try
judge                                   band                                   
S1 sat-trained                          [0.0, 0.5)          0.999         0.992
                                        [0.5, 0.8)          0.974         0.857
                                        [0.8, 0.9)          0.951         0.667
                                        [0.9, 0.95)         0.890         0.412
                                        [0.95, 0.98)        0.844         0.262
                                        [0.98, 1.0)         0.021         0.006
S5 sat-trained, relative + augmentation [0.0, 0.5)          0.999         0.994
                                        [0.5, 0.8)          0.977         0.875
                                        [0.8, 0.9)          0.949         0.667
                                        [0.9, 0.95)         0.923         0.381
                                        [0.95, 0.98)        0.870         0.297
                                        [0.98, 1.0)         0.023         0.004
reference: NUDT-trained (not shippable) [0.0, 0.5)          0.986         0.954
                                        [0.5, 0.8)          0.319         0.214
                                        [0.8, 0.9)          0.148         0.000
                                        [0.9, 0.95)         0.149         0.024
                                        [0.95, 0.98)        0.029         0.023
                                        [0.98, 1.0)         0.000         0.000

## 4. Ranking quality, and the cut on a finer grid near 1

The satellite-trained judges are over-confident on thermal candidates (section 3), so no cut of the standard grid (up to 0.99) passed
Learn-then-Test on NUDT and the rule answered nothing (the T10 'failed calibration -> no answers' fix). Only the ranking matters for a cut, so
the question is whether the ranking separates right from wrong on thermal photos (AUC), and whether a cut higher up the scale passes.

In [6]:
from sklearn.metrics import roc_auc_score
TAUS_FINE = cj.TAUS + [0.995, 0.998, 0.999, 0.9995, 0.9998, 0.9999]
rows = []
for var, m in [("pc", None), ("jetson", 5)]:
    J, _ = judges(m)
    kw = dict(min_choice=3, wide_tau=0.95, override=None, **(dict(jetson=True, m=5, early_tau=0.98) if m else {}))
    base = RESULTS[(var, "base")]
    for name, (ev, p) in J.items():
        L = cj.labelled(ev).to_numpy()
        auc = {g: round(roc_auc_score((ev.err_sigma <= 4).to_numpy()[L & sel], p[L & sel]), 4) for g, sel in
               [("NUDT design", ev.data.isin(NUDT_DESIGN).to_numpy()), ("official TRY", (ev.data == "official_try").to_numpy())]}
        dmask = ev.data.isin(NUDT_DESIGN).to_numpy()
        kws = {k: v for k, v in kw.items() if k != "early_tau"}
        tau = cj.ltt_tau(cj.scan_tau(ev[dmask], p[dmask], TAUS_FINE, **{**kws, "wide_tau": 0.0}), NUDT_DESIGN, 0.005)
        tau = cj.NO_ANSWER if tau is None else max(tau, 0.9)
        r = cj.decide(ev, p, tau, **{**kw, "wide_tau": tau, "early_tau": max(0.98, tau) if m else None})
        s = cj.summary(r)
        for d in ["official_try"] + NUDT_DESIGN:
            rows.append(dict(var=var, judge=name, auc_nudt=auc["NUDT design"], auc_official_try=auc["official TRY"], cut=tau, data=d,
                             score=s.loc[d, "score"], wrong=int(s.loc[d, "wrong"]), gain=cj.paired_gain(base, r, [d])[d]))
FINE = pd.DataFrame(rows); FINE.to_csv(R / "it1_sat_judge_fine_cut.csv", index=False)
FINE.pivot_table(index=["var", "judge", "auc_nudt", "auc_official_try", "cut"], columns="data", values=["score", "wrong"]).round(2)

score                                                wrong                                             
data                                                                           nudt101 nudt202 nudt404 nudt505 nudt606 official_try nudt101 nudt202 nudt404 nudt505 nudt606 official_try
var    judge                                   auc_nudt auc_official_try cut                                                                                                            
jetson S1 sat-trained                          0.9984   0.9963           0.998   54.16   55.20   56.76   53.52   54.98        51.54       1       0       0       1       0            0
       S5 sat-trained, relative + augmentation 0.9985   0.9973           0.995   54.36   55.38   55.56   55.40   55.30        52.15       1       0       2       0       0            0
       reference: NUDT-trained (not shippable) 0.9995   0.9990           inf      0.00    0.00    0.00    0.00    0.00         0.00       0       0       0       0       0            0
pc     S1 sat-trained                          0.9986   0.9965           0.998   54.84   56.16   56.22   55.36   56.60        54.03       1       1       0       0       0            0
       S5 sat-trained, relative + augmentation 0.9986   0.9974           0.998   54.64   55.86   56.22   55.96   57.00        53.71       1       0       0       0       0            0
       reference: NUDT-trained (not shippable) 0.9996   0.9989           0.925   55.52   56.76   56.52   56.96   56.40        54.03       0       0       0       0       1            0

## 5. The zero-wrong cut (T7 rule) on the fine grid

Learn-then-Test at alpha 0.005 tolerates a small wrong rate, and at 0.998 the satellite judges keep 1-2 wrong answers on the NUDT design sets
that set the cut. The team's rule is 0 wrong answers on every set, so the cut is instead the smallest one with 0 wrong answers on every NUDT
design set at it and at every stricter cut, plus one grid step (T7 `smallest_safe_tau`, margin 1).

In [7]:
rows, CUTS0 = [], {}
for var, m in [("pc", None), ("jetson", 5)]:
    J, _ = judges(m)
    base = RESULTS[(var, "base")]
    for name, (ev, p) in J.items():
        if name.startswith("reference"): continue
        kw = dict(min_choice=3, override=None, **(dict(jetson=True, m=5) if m else {}))
        dmask = ev.data.isin(NUDT_DESIGN).to_numpy()
        scan = pd.concat([cj.summary(cj.decide(ev[dmask], p[dmask], t, wide_tau=t, **kw)).reset_index().assign(tau=t) for t in TAUS_FINE])
        tau = cj.smallest_safe_tau(scan, NUDT_DESIGN, 1); CUTS0[(var, name)] = tau
        r = cj.decide(ev, p, tau, wide_tau=tau, early_tau=max(0.98, tau) if m else None, **kw)
        s = cj.summary(r)
        for d in ["official_try"] + NUDT_DESIGN:
            rows.append(dict(var=var, judge=name, cut=tau, data=d, score=s.loc[d, "score"], wrong=int(s.loc[d, "wrong"]), queries=s.loc[d, "queries"],
                             gain=cj.paired_gain(base, r, [d])[d]))
ZW = pd.DataFrame(rows); ZW.to_csv(R / "it1_sat_judge_zero_wrong_cut.csv", index=False)
display(ZW.pivot_table(index=["var", "judge", "cut"], columns="data", values=["score", "wrong"]).round(2))
ZW[ZW.data == "official_try"][["var", "judge", "cut", "score", "wrong", "queries", "gain"]]

score                                                wrong                                             
data                                                  nudt101 nudt202 nudt404 nudt505 nudt606 official_try nudt101 nudt202 nudt404 nudt505 nudt606 official_try
var    judge                                   cut                                                                                                             
jetson S1 sat-trained                          0.9995   52.64   53.82   55.20   52.96   53.18        48.71       0       0       0       0       0            0
       S5 sat-trained, relative + augmentation 0.9990   53.64   54.30   55.98   53.60   54.38        50.50       0       0       0       0       0            0
pc     S1 sat-trained                          0.9995   53.32   54.36   55.56   53.72   54.90        50.60       0       0       0       0       0            0
       S5 sat-trained, relative + augmentation 0.9995   53.72   54.66   55.56   54.12   55.08        51.57       0       0       0       0       0            0

,var,judge,cut,score,wrong,queries,gain
0,pc,S1 sat-trained,0.9995,50.595,0,9.859,"-0.78 [-2.40, +0.71]"
6,pc,"S5 sat-trained, relative + augmentation",0.9995,51.568,0,9.292,"+0.19 [-0.91, +1.36]"
12,jetson,S1 sat-trained,0.9995,48.714,0,3.070,"-2.66 [-4.67, -0.91]"
18,jetson,"S5 sat-trained, relative + augmentation",0.9990,50.497,0,2.589,"-0.88 [-2.04, +0.19]"
